In [1]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare
from scikit_posthocs import posthoc_nemenyi_friedman
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
import matplotlib.colors as mcolors
import os

# Define directories
base_dir = '/home/abidhasan/Document/Project/WaveFreqAug/results'
baseline_dir = os.path.join(base_dir, 'baseline')
wavefreq_dir = os.path.join(base_dir, 'wavefreq')
output_dir = "/home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis"
os.makedirs(output_dir, exist_ok=True)

# Define datasets and models
datasets = ['ETTh1', 'ETTh2', 'ILI', 'weather']
models = ['DLinear', 'SCINet']
pred_lens = {
    'ETTh1': [96, 192, 336, 720],
    'ETTh2': [96, 192, 336, 720],
    'ILI': [24, 36, 48, 60],
    'weather': [96, 192, 336, 720]
}

# Task 1: Find best WaveFreqAug results for each pred_len and save to CSV
for dataset in datasets:
    for model in models:
        avg_file = os.path.join(wavefreq_dir, model, f"average_results_{dataset}.csv")
        if os.path.exists(avg_file):
            df_avg = pd.read_csv(avg_file)
            # Group by pred_len and select rows with minimum MSE
            best_df = df_avg.loc[df_avg.groupby("pred_len")["mse"].idxmin()]
            best_file = os.path.join(output_dir, f"best_wavefreq_{dataset}_{model}.csv")
            best_df.to_csv(best_file, index=False)
            print(
                f"Saved best WaveFreqAug results for {dataset} {model} to {best_file}"
            )

Saved best WaveFreqAug results for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ETTh1_DLinear.csv
Saved best WaveFreqAug results for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ETTh1_SCINet.csv
Saved best WaveFreqAug results for ETTh2 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ETTh2_DLinear.csv
Saved best WaveFreqAug results for ETTh2 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ETTh2_SCINet.csv
Saved best WaveFreqAug results for ILI DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ILI_DLinear.csv
Saved best WaveFreqAug results for ILI SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/best_wavefreq_ILI_SCINet.csv
Saved best WaveFreqAug results for weather DLinear to /home/abidhasan/Document/Project/Wav

In [6]:
# Task 2: Create comparison tables and Line plot
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- IEEE-style aesthetic setup ---
plt.rcParams.update(
    {
        "font.size": 12,
        "font.family": "serif",
        "axes.linewidth": 1.2,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "legend.frameon": False,
        "figure.dpi": 800,
    }
)

# --- Define IEEE-like professional colors ---
color_map = {
    "Base": "#1f77b4",  # Deep Blue
    "StAug": "#ff7f0e",  # Orange
    "Freq-Mask": "#d62728",  # Red
    "Freq-Mix": "#9467bd",  # Muted Purple
    "Wave-Mask": "#8c564b",  # Brownish
    "Wave-Mix": "#17becf",  # Teal
    "WaveFreqAug": "#2ca02c",  # Green (highlight)
}

# --- Loop over datasets and models ---
plt.rcParams.update({"font.size": 12})
for dataset in datasets:
    for model in models:
        baseline_file = os.path.join(
            baseline_dir, model, f"average_results_{dataset}.csv"
        )
        best_wavefreq_file = os.path.join(
            output_dir, f"best_wavefreq_{dataset}_{model}.csv"
        )

        if os.path.exists(baseline_file) and os.path.exists(best_wavefreq_file):
            df_baseline = pd.read_csv(baseline_file)
            df_wavefreq = pd.read_csv(best_wavefreq_file)

            # Add missing columns to df_baseline
            for col in ["mask_rate", "level", "wavelet"]:
                df_baseline[col] = np.nan

            # Create comparison table
            df_comparison = pd.concat([df_baseline, df_wavefreq], ignore_index=True)
            comparison_file = os.path.join(
                output_dir, f"comparison_{dataset}_{model}.csv"
            )
            df_comparison.to_csv(comparison_file, index=False)
            print(f"Saved comparison table for {dataset} {model} to {comparison_file}")

            # --- Plot comparative analysis (Line plot with error bars for MSE per pred_len) ---
            pred_lens_dataset = pred_lens[dataset]
            aug_types = df_comparison["aug_type"].unique()

            fig, ax = plt.subplots(figsize=(10, 6))

            for aug_type in aug_types:
                df_aug = df_comparison[df_comparison["aug_type"] == aug_type]
                mse_values = [
                    (
                        df_aug[df_aug["pred_len"] == pl]["mse"].values[0]
                        if pl in df_aug["pred_len"].values
                        else np.nan
                    )
                    for pl in pred_lens_dataset
                ]
                mse_std_values = [
                    (
                        df_aug[df_aug["pred_len"] == pl]["mse_std"].values[0]
                        if pl in df_aug["pred_len"].values
                        else np.nan
                    )
                    for pl in pred_lens_dataset
                ]

                # Assign color from color_map
                color = color_map.get(aug_type, "gray")

                # Line plot with markers and error bars
                ax.errorbar(
                    pred_lens_dataset,
                    mse_values,
                    yerr=mse_std_values,
                    label=aug_type,
                    marker="o",
                    capsize=5,
                    linewidth=2,
                    color=color,
                )

            # --- Highlight WaveFreqAug line (bold, dashed, thicker) ---
            for line in ax.get_lines():
                if line.get_label() == "WaveFreqAug":
                    line.set_linewidth(3)
                    line.set_linestyle("--")
                    line.set_zorder(10)  # bring to front

            # --- Make markers visually cleaner ---
            for line in ax.get_lines():
                line.set_markerfacecolor("none")
                line.set_markersize(6)

            # --- Axis labels and ticks ---
            ax.set_xlabel("Prediction Length", fontsize=16)
            ax.set_ylabel("MSE", fontsize=16)
            ax.set_title(f"MSE Comparison for {dataset} with {model}", fontsize=16)
            ax.set_xticks(pred_lens_dataset)
            ax.set_xticklabels(pred_lens_dataset, fontsize=14)
            ax.tick_params(axis="y", labelsize=14)
            ax.grid(True, alpha=0.3)

            # --- Legend: WaveFreqAug on top and bold ---
            handles, labels = ax.get_legend_handles_labels()
            if "WaveFreqAug" in labels:
                wf_idx = labels.index("WaveFreqAug")
                # Move WaveFreqAug to first position
                handles.insert(0, handles.pop(wf_idx))
                labels.insert(0, labels.pop(wf_idx))
            legend = ax.legend(
                handles, labels, bbox_to_anchor=(1.05, 1), loc="upper left"
            )
            for text in legend.get_texts():
                if text.get_text() == "WaveFreqAug":
                    text.set_fontweight("bold")

            # --- Save both PNG and EPS ---
            for ext in ["png", "eps"]:
                plot_file = os.path.join(
                    output_dir, f"mse_comparison_lineplot_{dataset}_{model}.{ext}"
                )
                plt.savefig(plot_file, format=ext, dpi=300, bbox_inches="tight")
                print(f"Saved MSE line plot for {dataset} {model} to {plot_file}")

            plt.close()

Saved comparison table for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_DLinear.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved MSE line plot for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ETTh1_DLinear.png
Saved MSE line plot for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ETTh1_DLinear.eps
Saved comparison table for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_SCINet.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved MSE line plot for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ETTh1_SCINet.png
Saved MSE line plot for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ETTh1_SCINet.eps
Saved comparison table for ETTh2 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh2_DLinear.csv


...

Saved MSE line plot for ILI SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ILI_SCINet.png
Saved MSE line plot for ILI SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_ILI_SCINet.eps
Saved comparison table for weather DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_weather_DLinear.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved MSE line plot for weather DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_weather_DLinear.png
Saved MSE line plot for weather DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_weather_DLinear.eps
Saved comparison table for weather SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_weather_SCINet.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved MSE line plot for weather SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_weather_SCINet.png
Saved MSE line plot for weather SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_lineplot_weather_SCINet.eps


In [4]:
from scipy.ndimage import gaussian_filter1d
from math import pi

plt.rcParams.update({"font.size": 12})
sns.set_style("whitegrid")

for dataset in datasets:
    for model in models:
        baseline_file = os.path.join(
            baseline_dir, model, f"average_results_{dataset}.csv"
        )
        best_wavefreq_file = os.path.join(
            output_dir, f"best_wavefreq_{dataset}_{model}.csv"
        )

        if os.path.exists(baseline_file) and os.path.exists(best_wavefreq_file):
            df_baseline = pd.read_csv(baseline_file)
            df_wavefreq = pd.read_csv(best_wavefreq_file)

            # Add missing columns to baseline
            for col in ["mask_rate", "level", "wavelet"]:
                df_baseline[col] = np.nan

            # Combine baseline + best_wavefreq
            df_comparison = pd.concat([df_baseline, df_wavefreq], ignore_index=True)
            comparison_file = os.path.join(
                output_dir, f"comparison_{dataset}_{model}.csv"
            )
            df_comparison.to_csv(comparison_file, index=False)
            print(
                f"✅ Saved comparison table for {dataset} {model} to {comparison_file}"
            )

            # Define prediction lengths and colors
            pred_lens_dataset = pred_lens[dataset]
            color_map = {
                "Base": "blue",
                "StAug": "cyan",
                "Freq-Mask": "red",
                "Freq-Mix": "#665191",
                "Wave-Mask": "yellow",
                "Wave-Mix": "#a05195",
                "WaveFreqAug": "green",
            }

            # --- Prepare data for plotting ---
            df_plot = df_comparison[["aug_type", "pred_len", "mse", "mse_std"]].copy()
            df_plot = df_plot[df_plot["pred_len"].isin(pred_lens_dataset)]
            df_plot["color"] = df_plot["aug_type"].map(color_map)

            # ---------- (1) SMOOTHED LINE PLOT with markers ----------
            fig, ax = plt.subplots(figsize=(10, 6))
            for aug_type, group in df_plot.groupby("aug_type"):
                smooth_mse = gaussian_filter1d(group["mse"].values, sigma=1)
                sns.lineplot(
                    x=group["pred_len"],
                    y=smooth_mse,
                    label=aug_type,
                    color=color_map.get(aug_type, "gray"), # type: ignore
                    linewidth=2,
                    marker="o",
                    ax=ax,
                )
            ax.set_xlabel("Prediction Length", fontsize=14)
            ax.set_ylabel("MSE", fontsize=14)
            ax.set_title(f"{dataset} - {model}: Smoothed MSE Comparison", fontsize=16)
            ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            ax.grid(True, alpha=0.3)
            for ext in ["png", "eps"]:
                plt.savefig(
                    os.path.join(output_dir, f"mse_smoothed_{dataset}_{model}.{ext}"),
                    format=ext,
                    dpi=300,
                    bbox_inches="tight",
                )
            plt.close()

            # ---------- (2) STACKED AREA PLOT ----------
            pivot_df = df_plot.pivot(
                index="pred_len", columns="aug_type", values="mse"
            ).fillna(0)
            pivot_df = pivot_df.reindex(pred_lens_dataset)
            pivot_df.plot.area(
                color=[color_map.get(c, "gray") for c in pivot_df.columns],
                alpha=0.7,
                figsize=(10, 6),
            )
            plt.title(f"{dataset} - {model}: Stacked Area Plot of MSE")
            plt.xlabel("Prediction Length")
            plt.ylabel("MSE")
            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            for ext in ["png", "eps"]:
                plt.savefig(
                    os.path.join(output_dir, f"mse_area_{dataset}_{model}.{ext}"),
                    format=ext,
                    dpi=300,
                    bbox_inches="tight",
                )
            plt.close()

            # ---------- (3) FACET PLOT ----------
            g = sns.FacetGrid(
                df_plot, col="aug_type", col_wrap=3, height=3.5, sharey=False
            )
            g.map_dataframe(
                sns.lineplot, x="pred_len", y="mse", marker="o", linewidth=2
            )
            g.set_titles(col_template="{col_name}")
            g.set_axis_labels("Prediction Length", "MSE")
            plt.subplots_adjust(top=0.9)
            g.fig.suptitle(f"{dataset} - {model}: Faceted MSE Plot")
            for ext in ["png", "eps"]:
                g.savefig(
                    os.path.join(output_dir, f"mse_facet_{dataset}_{model}.{ext}"),
                    format=ext,
                    dpi=300,
                    bbox_inches="tight",
                )
            plt.close()

            # ---------- (4) RADAR / SPIDER CHART ----------
            radar_df = pivot_df.copy()
            radar_df = radar_df / radar_df.max()  # normalize for fair comparison
            categories = list(radar_df.index.astype(str))
            N = len(categories)
            angles = [n / float(N) * 2 * pi for n in range(N)]
            angles += angles[:1]

            fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
            for col in radar_df.columns:
                values = radar_df[col].values.tolist()
                values += values[:1]
                ax.plot(
                    angles,
                    values,
                    linewidth=2,
                    linestyle="--" if col == "WaveFreqAug" else "-",
                    label=col,
                    color=color_map.get(col, "gray"),
                )
                ax.fill(angles, values, alpha=0.15, color=color_map.get(col, "gray"))
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(categories)
            ax.set_title(
                f"{dataset} - {model}: Radar Chart (Normalized MSE)", size=14, y=1.08
            )
            ax.legend(bbox_to_anchor=(1.1, 1.1))
            for ext in ["png", "eps"]:
                plt.savefig(
                    os.path.join(output_dir, f"mse_radar_{dataset}_{model}.{ext}"),
                    format=ext,
                    dpi=300,
                    bbox_inches="tight",
                )
            plt.close()

            print(f"✅ Saved all 4 plot types for {dataset} {model}\n")

✅ Saved comparison table for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_DLinear.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


✅ Saved all 4 plot types for ETTh1 DLinear

✅ Saved comparison table for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_SCINet.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


✅ Saved all 4 plot types for ETTh1 SCINet

✅ Saved comparison table for ETTh2 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh2_DLinear.csv


...

✅ Saved all 4 plot types for ILI SCINet

✅ Saved comparison table for weather DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_weather_DLinear.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


✅ Saved all 4 plot types for weather DLinear

✅ Saved comparison table for weather SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_weather_SCINet.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


✅ Saved all 4 plot types for weather SCINet



In [3]:
# Task 2: Create comparison tables and plots
for dataset in datasets:
    for model in models:
        baseline_file = os.path.join(
            baseline_dir, model, f"average_results_{dataset}.csv"
        )
        best_wavefreq_file = os.path.join(
            output_dir, f"best_wavefreq_{dataset}_{model}.csv"
        )

        if os.path.exists(baseline_file) and os.path.exists(best_wavefreq_file):
            df_baseline = pd.read_csv(baseline_file)
            df_wavefreq = pd.read_csv(best_wavefreq_file)

            # Add missing columns to df_baseline
            for col in ["mask_rate", "level", "wavelet"]:
                df_baseline[col] = np.nan

            # Create comparison table
            df_comparison = pd.concat([df_baseline, df_wavefreq], ignore_index=True)
            comparison_file = os.path.join(
                output_dir, f"comparison_{dataset}_{model}.csv"
            )
            df_comparison.to_csv(comparison_file, index=False)
            print(f"Saved comparison table for {dataset} {model} to {comparison_file}")

            # Plot comparative analysis (Heatmap with magnified insets for MSE per pred_len)
            pred_lens_dataset = pred_lens[dataset]
            aug_types = df_comparison["aug_type"].unique()

            # Define specific colors for all methods
            color_map = {
                "Base": "blue",
                "STAug": "cyan",
                "Freq-Mask": "red",
                "Freq-Mix": "#665191",
                "Wave-Mask": "yellow",
                "Wave-Mix": "#a05195",
                "WaveFreqAug": "green",
            }

            # Prepare data for heatmap (normalized MSE as % difference from Base)
            df_base = df_comparison[df_comparison["aug_type"] == "Base"]
            base_mse = {
                pl: df_base[df_base["pred_len"] == pl]["mse"].values[0]
                for pl in pred_lens_dataset
                if pl in df_base["pred_len"].values
            }
            mse_matrix = np.zeros((len(aug_types), len(pred_lens_dataset)))
            mse_std_matrix = np.zeros((len(aug_types), len(pred_lens_dataset)))
            for i, aug_type in enumerate(aug_types):
                df_aug = df_comparison[df_comparison["aug_type"] == aug_type]
                mse_values = [
                    (
                        df_aug[df_aug["pred_len"] == pl]["mse"].values[0]
                        if pl in df_aug["pred_len"].values
                        else np.nan
                    )
                    for pl in pred_lens_dataset
                ]
                mse_std_values = [
                    (
                        df_aug[df_aug["pred_len"] == pl]["mse_std"].values[0]
                        if pl in df_aug["pred_len"].values
                        else np.nan
                    )
                    for pl in pred_lens_dataset
                ]
                mse_matrix[i, :] = [
                    (
                        100 * (mse - base_mse[pl]) / base_mse[pl]
                        if pl in base_mse and not np.isnan(mse)
                        else np.nan
                    )
                    for mse, pl in zip(mse_values, pred_lens_dataset)
                ]
                mse_std_matrix[i, :] = [
                    (
                        std / base_mse[pl] * 100
                        if pl in base_mse and not np.isnan(std)
                        else np.nan
                    )
                    for std, pl in zip(mse_std_values, pred_lens_dataset)
                ]

            # Create custom light gray colormap
            colors = [
                (0.95, 0.95, 0.95),
                (0.5, 0.5, 0.5),
            ]  # Very light gray to medium gray
            custom_cmap = mcolors.LinearSegmentedColormap.from_list(
                "CustomGray", colors
            )

            # Create figure
            fig, ax = plt.subplots(figsize=(10, 6))

            # Heatmap with custom light gray colormap and larger annotation font
            heatmap = sns.heatmap(
                mse_matrix,
                xticklabels=pred_lens_dataset,
                yticklabels=aug_types,
                cmap=custom_cmap,  # Custom very light gray colormap
                center=0,  # Center at 0% difference
                vmin=np.nanmin(mse_matrix) * 1.1,  # Adjust range to data
                vmax=np.nanmax(mse_matrix) * 1.1,
                annot=True,  # Show % difference values
                fmt=".2f",
                annot_kws={"size": 14},  # Increase font size of annotations
                cbar_kws={"label": "MSE (% Difference from Base)"},
                ax=ax,
            )
            # Set colorbar label padding using the existing colorbar
            heatmap.collections[0].colorbar.set_label(
                "MSE (% Difference from Base)", labelpad=10
            )

            # Highlight WaveFreqAug cells
            wavefreq_idx = (
                list(aug_types).index("WaveFreqAug")
                if "WaveFreqAug" in aug_types
                else -1
            )
            if wavefreq_idx >= 0:
                for j in range(len(pred_lens_dataset)):
                    ax.add_patch(
                        plt.Rectangle(
                            (j, wavefreq_idx), 1, 1, fill=False, edgecolor="black", lw=2
                        )
                    )

            ax.set_xlabel("Prediction Length", fontsize=16)
            ax.set_ylabel("Augmentation Method", fontsize=16)
            ax.set_title(f"MSE Comparison for {dataset} with {model}", fontsize=16)
            ax.tick_params(axis="both", labelsize=16)

            # Add insets for each prediction length
            for i, pl in enumerate(pred_lens_dataset):
                mse_values_all = [
                    (
                        df_comparison[df_comparison["pred_len"] == pl]["mse"].values[0]
                        if pl
                        in df_comparison[df_comparison["aug_type"] == aug_type][
                            "pred_len"
                        ].values
                        else np.nan
                    )
                    for aug_type in aug_types
                ]
                mse_std_all = [
                    (
                        df_comparison[df_comparison["pred_len"] == pl][
                            "mse_std"
                        ].values[0]
                        if pl
                        in df_comparison[df_comparison["aug_type"] == aug_type][
                            "pred_len"
                        ].values
                        else np.nan
                    )
                    for aug_type in aug_types
                ]
                mse_min, mse_max = np.nanmin(mse_values_all), np.nanmax(mse_values_all)
                if mse_min == mse_max or np.isnan(mse_min) or np.isnan(mse_max):
                    continue

                # Create inset axes
                inset_ax = inset_axes(
                    ax,
                    width=0.8,
                    height=1.2,
                    loc="upper center",
                    bbox_to_anchor=(0.15 + i * 0.2, 0.35, 0.2, 0.2),
                    bbox_transform=ax.transAxes,
                )
                for aug_type, mse, mse_std in zip(
                    aug_types, mse_values_all, mse_std_all
                ):
                    color = color_map.get(aug_type, "gray")
                    # Highlight WaveFreqAug in inset
                    if aug_type == "WaveFreqAug":
                        inset_ax.errorbar(
                            [aug_type],
                            [mse],
                            yerr=[mse_std],
                            fmt="o",
                            color=color,
                            capsize=3,
                            markersize=8,
                            linewidth=2,
                            linestyle="--",
                        )
                    else:
                        inset_ax.errorbar(
                            [aug_type],
                            [mse],
                            yerr=[mse_std],
                            fmt="o",
                            color=color,
                            capsize=3,
                        )

                # Set tight y-limits for inset
                margin = (mse_max - mse_min) * 0.1
                inset_ax.set_ylim(mse_min - margin, mse_max + margin)
                inset_ax.set_title(f"Pred. Len {pl}", fontsize=16)
                inset_ax.set_xticks([])
                inset_ax.grid(True, alpha=0.2)
                inset_ax.tick_params(axis="y", labelsize=16)

            # Save as both PNG and EPS
            for ext in ["png", "eps"]:
                plot_file = os.path.join(
                    output_dir, f"mse_comparison_heatmap_{dataset}_{model}.{ext}"
                )
                plt.savefig(plot_file, format=ext, dpi=300, bbox_inches="tight")
                print(f"Saved MSE heatmap for {dataset} {model} to {plot_file}")
            plt.close()

Saved comparison table for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_DLinear.csv
Saved MSE heatmap for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_heatmap_ETTh1_DLinear.png
Saved MSE heatmap for ETTh1 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_heatmap_ETTh1_DLinear.eps
Saved comparison table for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/comparison_ETTh1_SCINet.csv
Saved MSE heatmap for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_heatmap_ETTh1_SCINet.png
Saved MSE heatmap for ETTh1 SCINet to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/mse_comparison_heatmap_ETTh1_SCINet.eps
Saved comparison table for ETTh2 DLinear to /home/abidhasan/Document/Project/WaveFreqAug/result_analysis/analysis/compar

In [ ]:
# Task 3: Statistical tests on iteration results
for dataset in datasets:
    for model in models:
        baseline_file = os.path.join(
            baseline_dir, model, f"iteration_results_{dataset}.csv"
        )
        wavefreq_file = os.path.join(
            wavefreq_dir, model, f"iteration_results_{dataset}.csv"
        )

        if os.path.exists(baseline_file) and os.path.exists(wavefreq_file):
            df_baseline = pd.read_csv(baseline_file)
            df_wavefreq = pd.read_csv(wavefreq_file)

            # Combine baseline and WaveFreqAug results
            df_all = pd.concat([df_baseline, df_wavefreq])

            for pred_len in pred_lens[dataset]:
                # Filter for current pred_len
                df_pred = df_all[df_all["pred_len"] == pred_len]

                # Group by aug_type and collect MSE for each iteration
                aug_types = df_pred["aug_type"].unique()
                groups = []
                for aug_type in aug_types:
                    group = df_pred[df_pred["aug_type"] == aug_type]["mse"].values
                    if len(group) > 0:
                        groups.append(group)

                # Ensure all groups have the same number of iterations (3 for baselines, 5 for WaveFreqAug)
                if len(groups) > 1 and all(len(g) >= 3 for g in groups):
                    # Truncate to 3 iterations for all groups
                    groups_truncated = [g[:3] for g in groups]
                    # Friedman test
                    try:
                        friedman_stat, friedman_p = friedmanchisquare(*groups_truncated)
                        print(
                            f"{dataset} {model} pred_len={pred_len} Friedman test: stat={friedman_stat:.4f}, p={friedman_p:.4f}"
                        )

                        # Post-hoc Nemenyi test if Friedman is significant (p < 0.05)
                        if friedman_p < 0.05:
                            df_nemenyi = pd.DataFrame(groups_truncated).T
                            df_nemenyi.columns = aug_types
                            nemenyi = posthoc_nemenyi_friedman(df_nemenyi)
                            nemenyi_file = os.path.join(
                                output_dir, f"nemenyi_{dataset}_{model}_{pred_len}.csv"
                            )
                            nemenyi.to_csv(nemenyi_file)
                            print(
                                f"Saved Nemenyi test for {dataset} {model} pred_len={pred_len} to {nemenyi_file}"
                            )
                    except ValueError as e:
                        print(
                            f"Error in Friedman test for {dataset} {model} pred_len={pred_len}: {e}"
                        )

ETTh1 DLinear pred_len=96 Friedman test: stat=14.4286, p=0.0252
Saved Nemenyi test for ETTh1 DLinear pred_len=96 to /home/abidhasan/Document/Project/WaveFreqAug/results/analysis/nemenyi_ETTh1_DLinear_96.csv
ETTh1 DLinear pred_len=192 Friedman test: stat=8.8571, p=0.1818
ETTh1 DLinear pred_len=336 Friedman test: stat=5.2857, p=0.5077
ETTh1 DLinear pred_len=720 Friedman test: stat=11.2857, p=0.0799
ETTh1 SCINet pred_len=96 Friedman test: stat=11.4286, p=0.0760
ETTh1 SCINet pred_len=192 Friedman test: stat=9.1429, p=0.1657
ETTh1 SCINet pred_len=336 Friedman test: stat=9.7143, p=0.1372
ETTh1 SCINet pred_len=720 Friedman test: stat=10.2857, p=0.1131
ETTh2 DLinear pred_len=96 Friedman test: stat=16.1429, p=0.0130
Saved Nemenyi test for ETTh2 DLinear pred_len=96 to /home/abidhasan/Document/Project/WaveFreqAug/results/analysis/nemenyi_ETTh2_DLinear_96.csv
ETTh2 DLinear pred_len=192 Friedman test: stat=14.0000, p=0.0296
Saved Nemenyi test for ETTh2 DLinear pred_len=192 to /home/abidhasan/Docume